# Chapter 22 — The 2026 frontier: sparse, windowed and linear attention

Every number in Chapter 22, recomputed from `tfs/linear_attention.py` and `tfs/foundations.py`, plus
small working versions of the mechanisms the chapter describes: a top-k sparse-attention layer in the
style of DeepSeek Sparse Attention, a sliding-window mask, a softmax with a sink, linear attention in its
two forms, the delta rule as a gradient step, the chunked training algorithm, and the memory and FLOP
arithmetic of hybrids. Each cell **asserts** what it checks, so `python tests/run_notebooks.py 22` fails
if the chapter and the code disagree.

Shapes are written for **one head**: `q, k : (T, dk)`, `v : (T, dv)`, the state `S : (dv, dk)`.

In [ ]:
import math, sys, os
sys.path.insert(0, os.path.abspath(".."))      # run from notebooks/
sys.path.insert(0, os.path.abspath("."))       # or from the repo root
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)
torch.set_num_threads(min(4, torch.get_num_threads()))
import tfs.linear_attention as LA
from tfs.foundations import softmax_with_sink
def qkvba(T=100, dk=16, dv=8, seed=0):
    g = torch.Generator().manual_seed(seed)
    q = torch.randn(T, dk, generator=g, dtype=torch.float64)                      # (T, dk)
    k = F.normalize(torch.randn(T, dk, generator=g, dtype=torch.float64), dim=-1)   # (T, dk) unit keys
    v = torch.randn(T, dv, generator=g, dtype=torch.float64)                      # (T, dv)
    b = torch.rand(T, generator=g, dtype=torch.float64)                            # (T,)  write strength
    a = torch.rand(T, generator=g, dtype=torch.float64) * 0.5 + 0.5                # (T,)  forget gate
    return q, k, v, b, a

## 22.1 — sparse attention: index cheaply, attend to the top k

A toy of DSA's two stages. A cheap *indexer* (here: a 4-wide projection of q and k, ReLU-gated) scores
every past position; real attention then runs over only the `k` best. Two checks: with `k = T` the
layer **is** dense causal attention, and the expensive stage's score count falls from `T(T+1)/2` to
about `T·k`.

In [ ]:
torch.manual_seed(0)
T, d, d_idx, topk = 256, 32, 4, 32
q, k, v = torch.randn(T, d), torch.randn(T, d), torch.randn(T, d)          # (T, d) each
Wq_i, Wk_i = torch.randn(d, d_idx), torch.randn(d, d_idx)                   # the indexer's projections
causal = torch.ones(T, T, dtype=torch.bool).tril()                          # (T, T)

def dense(q, k, v):
    s = (q @ k.T) / math.sqrt(d)                                            # (T, T)
    return s.masked_fill(~causal, float("-inf")).softmax(-1) @ v            # (T, d)

def sparse(q, k, v, topk):
    idx = F.relu((q @ Wq_i) @ (k @ Wk_i).T)                                 # (T, T)  cheap scores
    idx = idx.masked_fill(~causal, float("-inf"))
    kk = min(topk, T)
    sel = idx.topk(kk, dim=-1).indices                                      # (T, kk) chosen positions
    keep = torch.zeros(T, T, dtype=torch.bool).scatter_(1, sel, True) & causal
    s = (q @ k.T) / math.sqrt(d)
    return s.masked_fill(~keep, float("-inf")).softmax(-1) @ v, keep.sum().item()

full, _ = sparse(q, k, v, T)
assert torch.allclose(full, dense(q, k, v), atol=1e-5)                    # k = T recovers dense attention
out, n_scores = sparse(q, k, v, topk)
dense_scores = T * (T + 1) // 2
print(f"expensive scores: dense {dense_scores:,}  top-{topk} {n_scores:,}  ({dense_scores / n_scores:.1f}x fewer)")
assert n_scores <= T * topk and dense_scores / n_scores > 4                 # O(L^2) -> O(L k)

## 22.2 — windows: the KV cache that stops growing

gpt-oss-120b: 36 layers, half of them banded to 128 tokens, 8 KV heads of width 64, bf16. The chapter's
numbers, then a mask check that a window layer really reads only the last `w` tokens.

In [ ]:
per_tok = LA.kv_cache_bytes_per_token(18, 8, 64)                 # the 18 full layers
window = per_tok * 128                                            # the 18 window layers, capped
T = 131_072
alternating, all_full = per_tok * T + window, 2 * per_tok * T
print(f"per token {per_tok:,} B  window total {window / 1e6:.1f} MB  "
      f"alternating {alternating / 1e9:.2f} GB  all-full {all_full / 1e9:.2f} GB")
assert per_tok == 36_864 and window == 4_718_592
assert round(per_tok * T / 1e9, 2) == 4.83 and round(all_full / 1e9, 2) == 9.66
assert abs(alternating / all_full - 0.5) < 0.001                  # "halves the cache, almost exactly"

w, n = 4, 10
i = torch.arange(n)
band = (i[:, None] >= i[None, :]) & (i[:, None] - i[None, :] < w) # (n, n) query i sees keys i-w+1..i
assert band.sum(-1).tolist() == [1, 2, 3, 4, 4, 4, 4, 4, 4, 4]    # cost O(w), not O(T)

## 22.2 — the worked example: a sink logit lets a head attend to nothing

In [ ]:
scores = [2.0, 1.0, 0.5]
none = torch.tensor(scores, dtype=torch.float64).softmax(-1)
s0, s4 = softmax_with_sink(scores, 0.0), softmax_with_sink(scores, 4.0)
print("no sink", none.numpy().round(4), " s=0", s0.numpy().round(4), round(s0.sum().item(), 4),
      " s=4 sum", round(s4.sum().item(), 4))
assert [round(x, 4) for x in none.tolist()] == [0.6285, 0.2312, 0.1402]
assert [round(x, 4) for x in s0.tolist()] == [0.5793, 0.2131, 0.1293] and round(s0.sum().item(), 4) == 0.9216
assert round(s4.sum().item(), 4) == 0.1772
# s = 0 is the "off-by-one" softmax: e^x / (1 + sum e^x)
e = torch.tensor(scores, dtype=torch.float64).exp()
assert torch.allclose(s0, e / (1 + e.sum()))
# and the order of real weights never changes: the sink only scales them down
assert torch.allclose(s4 / s4.sum(), none)

## 22.3 — linear attention: one recurrence, one masked matmul

Drop the softmax and attention becomes `o = ((Q Kᵀ) ⊙ causal) V`. Reassociate and it is a recurrence
on a `(dv, dk)` state: write `S += v kᵀ`, read `S q`. Both forms, equal to float64 precision.

In [ ]:
q, k, v, _, _ = qkvba()
o_rec, S = LA.linear_attention_recurrent(q, k, v)                # (T, dv), (dv, dk)
o_par = LA.linear_attention_parallel(q, k, v)                    # (T, dv)
print("state", tuple(S.shape), " max |rec - par|", (o_rec - o_par).abs().max().item())
assert S.shape == (8, 16) and torch.allclose(o_rec, o_par, atol=1e-10)
# the state is the sum of outer products: memory constant in T
assert torch.allclose(S, v.T @ k)

## 22.4 — the delta rule: write by correcting the error

The 2-D worked example: write `(k₁, v₁)`, `(k₂, v₂)`, then a **new** value for `k₁`. Linear attention
returns the sum of both `k₁` values; the delta rule returns the new one. Then the identity the chapter
builds on: one delta-rule write is one SGD step on `½‖S k − v‖²` with learning rate β.

In [ ]:
lin, delta, S_lin, S_delta = LA.overwrite_demo()
print("read k1 -> linear", lin, " delta", delta)
assert lin == [2.0, 5.0] and delta == [-1.0, 4.0]
assert S_lin == [[2.0, 0.0], [5.0, 2.0]] and S_delta == [[-1.0, 0.0], [4.0, 2.0]]

torch.manual_seed(0)
S0 = torch.randn(4, 3, dtype=torch.float64, requires_grad=True)          # (dv, dk)
kk = F.normalize(torch.randn(3, dtype=torch.float64), dim=0)
vv, beta = torch.randn(4, dtype=torch.float64), 0.7
(0.5 * (S0 @ kk - vv).pow(2).sum()).backward()
sgd = S0.detach() - beta * S0.grad
_, S1 = LA.delta_rule_recurrent(kk[None], kk[None], vv[None],
                                torch.tensor([beta], dtype=torch.float64), S0=S0.detach())
assert torch.allclose(sgd, S1, atol=1e-12)
print("delta write == one SGD step: True")

## 22.4 — what the state can remember

Associative recall with one 64-wide head. With orthogonal keys both rules are exact up to `dk` pairs.
With random unit keys, linear attention's error grows like `√((n−1)/dk)`; the delta rule degrades far
more slowly; softmax attention, which keeps every pair, stays exact.

In [ ]:
for n in (8, 32, 64):
    assert LA.recall_error(n, 64, "linear", "orthogonal") < 1e-10
    assert LA.recall_error(n, 64, "delta", "orthogonal") < 1e-10
print(f"{'n':>5} {'linear':>8} {'sqrt law':>9} {'delta':>7} {'softmax':>8}")
lin = {}
for n in (8, 64, 256, 1024):
    lin[n] = LA.recall_error(n, 64, "linear")
    print(f"{n:5d} {lin[n]:8.3f} {LA.interference_std(n, 64):9.3f} "
          f"{LA.recall_error(n, 64, 'delta'):7.3f} {LA.recall_error(n, 64, 'softmax'):8.1e}")
assert [round(lin[n], 3) for n in (8, 64, 256, 1024)] == [0.278, 1.022, 1.983, 4.005]
for n in (64, 256, 1024):
    assert abs(lin[n] / LA.interference_std(n, 64) - 1) < 0.04
assert round(LA.recall_error(1024, 64, "delta"), 3) == 1.349
assert LA.recall_error(1024, 64, "softmax") < 1e-6

### Exercise — recall by age (dk = 32)

The square-root law's predictions, the measured linear-attention errors, and the delta rule's bias
toward the recent past: the newest tenth of 512 pairs is recalled better than the oldest tenth.

In [ ]:
for n, pred in ((32, 0.984), (128, 1.992), (512, 3.996)):
    assert round(LA.interference_std(n, 32), 3) == pred
assert round(LA.recall_error(128, 32, "linear"), 3) == 2.130
assert round(LA.recall_error(512, 32, "linear"), 3) == 3.989
old, new = LA.recall_error_by_age(512, 32, "delta")
lo, ln_ = LA.recall_error_by_age(512, 32, "linear")
print(f"delta  oldest {old:.3f}  newest {new:.3f}     linear  oldest {lo:.3f}  newest {ln_:.3f}")
assert (round(old, 3), round(new, 3)) == (1.460, 0.970) and new < old      # recency-weighted
assert (round(lo, 3), round(ln_, 3)) == (3.925, 4.014)                     # linear: no recency at all

## 22.5 — the chunked form equals the recurrence

Training cannot loop over 100k tokens. Within a chunk of `C` tokens the delta rule is a few matmuls and
one triangular solve; only the chunk-to-chunk state is sequential. Every chunk size, gated and ungated,
must give the recurrence's outputs **and** final state.

In [ ]:
q, k, v, b, a = qkvba()
for C in (1, 7, 16, 64, 100):
    o1, S1 = LA.gated_delta_rule_recurrent(q, k, v, b, a)
    o2, S2 = LA.chunked_delta_rule(q, k, v, b, a, chunk=C)
    assert torch.allclose(o1, o2, atol=1e-10) and torch.allclose(S1, S2, atol=1e-10)
    o1, _ = LA.delta_rule_recurrent(q, k, v, b)
    o2, _ = LA.chunked_delta_rule(q, k, v, b, chunk=C)
    assert torch.allclose(o1, o2, atol=1e-10)
print("chunked == recurrent for C in 1, 7, 16, 64, 100 (gated and ungated)")

f_lin = LA.chunked_delta_flops_per_token(64, 128, 128)
f_4k = LA.softmax_attention_flops_per_token(4096, 128, 128)
f_256k = LA.softmax_attention_flops_per_token(262_144, 128, 128)
print(f"FLOPs/token: chunked delta {f_lin:,}   softmax @4k {f_4k:,.0f}   @256k {f_256k:,.0f}")
assert (f_lin, f_4k, f_256k) == (155_648, 1_048_576, 67_108_864)
# softmax attention's per-token cost passes the delta rule's at a few hundred tokens
cross = next(T for T in range(2, 10_000) if LA.softmax_attention_flops_per_token(T, 128, 128) > f_lin)
print("crossover at T =", cross)
assert 500 < cross < 700

## 22.6 — hybrids: the memory arithmetic

Qwen3-Next-80B-A3B: 12 gated-attention layers (2 KV heads × 256) and 36 Gated DeltaNet layers (32 heads,
128 × 128 state, fp32). At a 262,144-token context the hybrid needs about a quarter of the memory of the
same model with attention in every layer. Exercise 22.5's 1 : 3 design breaks even at 512 tokens.

In [ ]:
assert LA.kv_cache_bytes_per_token(12, 2, 256) == 24_576
assert LA.linear_state_bytes(36, 32, 128, 128) == 75_497_472
hyb, dense_b = LA.qwen3_next_memory(262_144)
print(f"hybrid {hyb / 1e9:.3f} GB   all-attention {dense_b / 1e9:.3f} GB   ratio {hyb / dense_b:.3f}")
assert round(hyb / 1e9, 3) == 6.518 and round(dense_b / 1e9, 3) == 25.770
# the delta state is FIXED: at short context the hybrid is the more expensive one
short_h, short_d = LA.qwen3_next_memory(512)
assert short_h > short_d
even_h, even_d = LA.qwen3_next_memory(1024)
assert even_h == even_d                     # break-even at exactly 1,024 tokens

# Exercise 22.5: 64 attention layers vs 16 attention + 48 delta layers, 8 KV heads x 128
full, hybrid_kv = LA.kv_cache_bytes_per_token(64, 8, 128), LA.kv_cache_bytes_per_token(16, 8, 128)
state = LA.linear_state_bytes(48, 32, 128, 128)
assert (full, hybrid_kv, state) == (262_144, 65_536, 100_663_296)
print("break-even context:", state // (full - hybrid_kv), "tokens")
assert state / (full - hybrid_kv) == 512

## Summary

| claim | checked |
|---|---|
| top-k sparse attention with k = T is dense attention; k ≪ T cuts the expensive scores to ~T·k | §22.1 |
| gpt-oss alternation: 36,864 B/token, 4.7 MB of window cache, 4.83 vs 9.66 GB at 131k | §22.2 |
| sink weights 0.6285/0.2312/0.1402 → 0.9216 total at s = 0 → 0.1772 at s = 4 | §22.2 |
| linear attention's recurrent and parallel forms agree | §22.3 |
| the delta rule overwrites; one write is one SGD step | §22.4 |
| recall error √((n−1)/dk) for linear attention; the delta rule is recency-weighted | §22.4 |
| the chunked algorithm equals the recurrence; 155,648 FLOPs/token, independent of T | §22.5 |
| Qwen3-Next: 6.518 vs 25.770 GB at 262k; the 1 : 3 hybrid breaks even at 512 tokens | §22.6 |